### Виды тестирования и их применение

| Вид тестирования | Описание | Применение |
|-----------------|----------|------------|
| **Модульное** *(Unit testing)* | Проверяет отдельные функции/методы в **изоляции** | Основной тип тестирования |
| **Интеграционное** *(Integration testing)* | Проверяет взаимодействие модулей | Тестирование работы сервиса с базой данных |
| **Функциональное** *(Functional testing)* | Проверяет работу приложения с точки зрения пользователя | API-тесты, UI-тесты |
| **Регрессионное** *(Regression testing)* | Убеждается, что изменения не сломали существующий код | Тесты перед деплоем |
| **Нагрузочное** *(Load testing)* | Проверяет, как система ведёт себя под нагрузкой | Тестирование сервера на 1000+ запросов |
| **Приемочное** *(Acceptance testing)* | Проверяет соответствие требованиям бизнеса | Финальные тесты перед продакшеном |


### Модульное тестирование (`unittest`, `pytest`)

Модульные тесты изолируют отдельные компоненты системы. Они **не должны зависеть** от БД, сети и других внешних сервисов.

In [ ]:
assert 1 == 2, "Not equal"

AssertionError: Not equal

In [ ]:
import unittest

def add(a, b):
    return a + b

class TestMath(unittest.TestCase):

    def test_add(self):
        self.assertEqual(add(2, 3), 5)
        self.assertEqual(add(-1, 1), 0)

    def test_type_error(self):
        with self.assertRaises(TypeError):
            add(2, "three")

if __name__ == "__main__":
    unittest.main()

E
ERROR: /root/ (unittest.loader._FailedTest./root/)
----------------------------------------------------------------------
AttributeError: module '__main__' has no attribute '/root/'

----------------------------------------------------------------------
Ran 1 test in 0.006s

FAILED (errors=1)


SystemExit: True

/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### 1. `unittest`: встроенный инструмент
```python
import unittest

def add(a, b):
    return a + b

class TestMath(unittest.TestCase):

    def test_add(self):
        self.assertEqual(add(2, 3), 5)
        self.assertEqual(add(-1, 1), 0)

    def test_type_error(self):
        with self.assertRaises(TypeError):
            add(2, "three")

if __name__ == "__main__":
    unittest.main()
```
**Минусы `unittest`**:
- Много шаблонного кода.
- Слабая интеграция с `pytest` и асинхронным кодом.

### 2. `pytest`: современный стандарт
`pytest` удобнее `unittest`, так как:
- Использует `assert`, а не `self.assertEqual()`.
- Поддерживает фикстуры и параметризацию.

```python
import pytest

def add(a, b):
    return a + b

@pytest.mark.parametrize("a, b, expected", [
    (2, 3, 5),
    (-1, 1, 0),
    (10, -10, 0)
])
def test_add(a, b, expected):
    assert add(a, b) == expected
```


```bash
pytest test_math.py -v -s
```

In [ ]:
import pytest

@pytest.fixture
def sample_data():
    return {"name": "Alice", "age": 30}

def test_data_processing(sample_data):
    assert sample_data["name"] == "Alice"
    assert sample_data["age"] == 30


### Интеграционное тестирование (работа с БД, API)

### 1. Подключение к базе данных
При тестировании базы **нельзя использовать продакшен-данные**. Лучшие практики:
1. **In-memory БД** (`sqlite:///:memory:`).
2. **Фикстуры для создания тестовых данных**.
3. **Откат транзакций после каждого теста**.

#### **Пример с `SQLAlchemy`**
```python
from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy.orm import declarative_base, sessionmaker
import pytest

Base = declarative_base()

class User(Base):
    __tablename__ = "users"
    id = Column(Integer, primary_key=True)
    name = Column(String)

@pytest.fixture
def db_session():
    engine = create_engine("sqlite:///:memory:")
    Base.metadata.create_all(engine)
    Session = sessionmaker(bind=engine)
    session = Session()
    yield session
    session.close()

def test_user_creation(db_session):
    user = User(name="Alice")
    db_session.add(user)
    db_session.commit()

    result = db_session.query(User).first()
    assert result.name == "Alice"
```

Что важно помнить?
- **Не использовать реальную базу** для тестов.
- **Откатывать изменения** после тестов (`session.rollback()`).

### 2. Тестирование API (`requests`, `FastAPI`)
API тестируется на уровне HTTP-запросов. Можно использовать `requests` или `httpx`.

```python
import requests

def test_api():
    response = requests.get("https://jsonplaceholder.typicode.com/todos/1")
    assert response.status_code == 200
    assert response.json()["id"] == 1
```

**Тестирование FastAPI-приложения**:
```python
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()

@app.get("/hello")
def hello():
    return {"message": "Hello, world!"}

client = TestClient(app)

def test_hello():
    response = client.get("/hello")
    assert response.status_code == 200
    assert response.json() == {"message": "Hello, world!"}
```

## Мокирование зависимостей (`mock`, `pytest-mock`)

Мокирование (Mocking) позволяет заменить реальные сервисы **фейковыми объектами**.

```python
from unittest.mock import MagicMock

class APIClient:
    def fetch_data(self):
        raise NotImplementedError()

def test_mock():
    mock_client = MagicMock()
    mock_client.fetch_data.return_value = {"status": "ok"}

    assert mock_client.fetch_data() == {"status": "ok"}
```

In [ ]:
# Допустим, у нас есть функция, которая получает данные погоды через внешний API
import requests

def get_weather_data(city):
    url = f"https://api.weather.example.com/data?city={city}"
    response = requests.get(url)
    return response.json()

# В тесте мы подменим requests.get мок-объектом, чтобы не делать реальный HTTP запрос
from unittest.mock import MagicMock, patch

def test_get_weather_data():
    # Создаем мок-объект ответа
    mock_response = MagicMock()
    mock_response.json.return_value = {"temperature": "20°C", "humidity": "60%"}
    # Патчим requests.get, чтобы он возвращал наш мок-объект вместо реального вызова
    with patch('requests.get', return_value=mock_response):
        data = get_weather_data("London")
        # Проверяем, что функция вернула ожидаемые данные
        assert data == {"temperature": "20°C", "humidity": "60%"}
        # Дополнительно убеждаемся, что requests.get вызывался ровно один раз с нужным URL
        requests.get.assert_called_once_with("https://api.weather.example.com/data?city=London")


## Асинхронное тестирование (`pytest-asyncio`)

Асинхронный код требует `pytest-asyncio`.

```python
import pytest
import asyncio

@pytest.mark.asyncio
async def test_async():
    async def async_task():
        await asyncio.sleep(1)
        return 42

    result = await async_task()
    assert result == 42
```

### Производительность тестов (`pytest-benchmark`)

Чтобы измерять производительность функций, можно использовать `pytest-benchmark`:
```bash
pip install pytest-benchmark
```

```python
import time

def slow_function():
    time.sleep(1)
    return 42

def test_slow_function(benchmark):
    result = benchmark(slow_function)
    assert result == 42
```


In [ ]:
from locust import HttpUser, between, task

# Определяем класс пользователя, наследуясь от HttpUser
class APIUser(HttpUser):
    host = "http://example.com"            # базовый URL тестируемого API
    wait_time = between(1, 5)             # пауза между запросами от 1 до 5 секунд

    @task
    def list_items(self):
        # Задача: получить список элементов
        self.client.get("/api/items")


### Покрытие кода тестами (`coverage.py`)

Чтобы узнать, какие части кода не покрыты тестами:
```bash
pip install coverage
coverage run -m pytest
coverage report
coverage html  # Генерация отчёта в HTML
```